In [0]:
storage_account = "retailpulsestorageacc"
container = 'retail-data'
base_path = (
    f"abfss://{container}@{storage_account}"
    ".dfs.core.windows.net"
)

raw_path = f"{base_path}/bronze/github"
delta_path = f"{base_path}/bronze/delta" 


In [0]:
display(dbutils.fs.ls(raw_path))

path,name,size,modificationTime
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/,customers/,0,1792911733000
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/products/,products/,0,1792911757000
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales/,sales/,0,1792911711000
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/stores/,stores/,0,1792911777000


In [0]:
display(dbutils.fs.ls(f"{raw_path}/sales"))

path,name,size,modificationTime
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales/sales.csv,sales.csv,78161,1792913771000


## Read Raw CSV Files

#### Read Raw CSV File

In [0]:
sales_path = f"{raw_path}/sales"

sales_df = (
    spark.read
    .format('csv')
    .option('header', "true")
    .option('inferschema', 'false')
    .load(sales_path)
)

In [0]:
def read_csv(dataset_name):
    file_path = f"{raw_path}/{dataset_name}/{dataset_name}.csv"

    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferschema", "false")
        .load(file_path)
    )
    

customers_df = read_csv("customers")
products_df = read_csv("products")
stores_df = read_csv("stores")

In [0]:
print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Stores:", stores_df.count())
print("Sales:", sales_df.count())

Customers: 200
Products: 50
Stores: 10
Sales: 1000


In [0]:
display(sales_df.limit(10))
display(customers_df.limit(10))
display(products_df.limit(10))
display(stores_df.limit(10))

order_id,order_date,customer_id,product_id,store_id,quantity,unit_price,discount,status,last_modified_date
O00001,2025-01-30,C0111,P008,S10,4,2499.00,0.00,Completed,2025-01-30T19:19:00
O00002,2025-11-22,C0069,P012,S04,4,2499.00,0.00,Completed,2025-11-22T09:42:00
O00003,2025-12-02,C0158,P044,S06,5,299.00,0.00,Cancelled,2025-12-02T03:01:00
O00004,2025-08-12,C0043,P037,S09,1,299.00,0.00,Completed,2025-08-12T15:26:00
O00005,2025-10-11,C0200,P022,S07,1,799.00,0.00,Completed,2025-10-11T22:54:00
O00006,2025-07-20,C0023,P010,S02,4,4999.00,100.00,Completed,2025-07-20T05:17:00
O00007,2025-08-31,C0152,P012,S01,3,2499.00,100.00,Completed,2025-08-31T23:58:00
O00008,2025-01-28,C0076,P010,S04,5,4999.00,0.00,Completed,2025-01-28T09:15:00
O00009,2025-10-01,C0174,P040,S01,5,2499.00,0.00,Completed,2025-10-01T12:05:00
O00010,2025-06-14,C0091,P029,S05,2,2499.00,0.00,Completed,2025-06-14T00:43:00


customer_id,customer_name,city,region,registration_date
C0001,Customer 001,Hyderabad,South,2025-03-10
C0002,Customer 002,Bengaluru,South,2025-01-23
C0003,Customer 003,Chennai,South,2024-08-18
C0004,Customer 004,Mumbai,West,2024-07-19
C0005,Customer 005,Pune,West,2025-08-08
C0006,Customer 006,Delhi,North,2025-03-30
C0007,Customer 007,Jaipur,North,2024-03-14
C0008,Customer 008,Kolkata,East,2025-02-22
C0009,Customer 009,Bhubaneswar,East,2025-01-07
C0010,Customer 010,Ahmedabad,West,2024-05-27


product_id,product_name,category,list_price
P001,Electronics Item 01,Electronics,199.00
P002,Electronics Item 02,Electronics,299.00
P003,Electronics Item 03,Electronics,299.00
P004,Electronics Item 04,Electronics,1299.00
P005,Electronics Item 05,Electronics,799.00
P006,Electronics Item 06,Electronics,799.00
P007,Electronics Item 07,Electronics,299.00
P008,Electronics Item 08,Electronics,2499.00
P009,Electronics Item 09,Electronics,4999.00
P010,Electronics Item 10,Electronics,4999.00


store_id,store_name,city,region,store_type
S01,RetailPulse Hyderabad,Hyderabad,South,Mall
S02,RetailPulse Bengaluru,Bengaluru,South,High Street
S03,RetailPulse Chennai,Chennai,South,Mall
S04,RetailPulse Mumbai,Mumbai,West,High Street
S05,RetailPulse Pune,Pune,West,Mall
S06,RetailPulse Delhi,Delhi,North,High Street
S07,RetailPulse Jaipur,Jaipur,North,Mall
S08,RetailPulse Kolkata,Kolkata,East,High Street
S09,RetailPulse Bhubaneswar,Bhubaneswar,East,Mall
S10,RetailPulse Ahmedabad,Ahmedabad,West,High Street


### Adding metadata columns to the raw CSV files

In [0]:
import pyspark.sql.functions as F
from datetime import datetime, timezone
import uuid

batch_id = str(uuid.uuid4())
batch_timestamp = datetime.now(timezone.utc)

def add_metadata(df):
    return (
        df
        .withColumn(
            "_source_file", 
            F.col("_source_file") 
            if "_source_file" in df.columns
            else F.lit(None).cast("string")
        )
        .withColumn(
            "_ingestion_timestamp", 
            F.lit(batch_timestamp).cast("timestamp")
        )
        .withColumn(
            "_ingestion_batch_id",
            F.lit(batch_id)
        )
    )

In [0]:
def prepare_bronze(df, source_path):
    return add_metadata(
        df.withColumn("_source_file", F.lit(source_path))
    )


sales_bronze = prepare_bronze(sales_df, sales_path)

customers_bronze = prepare_bronze(
    customers_df, 
    f"{raw_path}/customers/customers.csv"
)

products_bronze = prepare_bronze(
    products_df,
    f"{raw_path}/products/products.csv"
)

stores_bronze = prepare_bronze(
    stores_df, 
    f"{raw_path}/stores/stores.csv"
)

In [0]:
display(sales_bronze.limit(5))

order_id,order_date,customer_id,product_id,store_id,quantity,unit_price,discount,status,last_modified_date,_source_file,_ingestion_timestamp,_ingestion_batch_id
O00001,2025-01-30,C0111,P008,S10,4,2499.00,0.00,Completed,2025-01-30T19:19:00,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
O00002,2025-11-22,C0069,P012,S04,4,2499.00,0.00,Completed,2025-11-22T09:42:00,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
O00003,2025-12-02,C0158,P044,S06,5,299.00,0.00,Cancelled,2025-12-02T03:01:00,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
O00004,2025-08-12,C0043,P037,S09,1,299.00,0.00,Completed,2025-08-12T15:26:00,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
O00005,2025-10-11,C0200,P022,S07,1,799.00,0.00,Completed,2025-10-11T22:54:00,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125


## Creating and Storing Delta Tables from the raw CSV Files 

In [0]:
sales_delta_path = f"{delta_path}/sales"

(
    sales_bronze.write
    .format("delta")
    .mode("overwrite")
    .save(sales_delta_path)
)

In [0]:
def write_bronze_delta(df, dataset_name):
    destination = f"{delta_path}/{dataset_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(destination)
    )

    print(f"completed: {dataset_name}")
    print(f"saved to: {destination}")

In [0]:
write_bronze_delta(sales_bronze, "sales")
write_bronze_delta(customers_bronze, "customers")
write_bronze_delta(products_bronze, "products")
write_bronze_delta(stores_bronze, "stores")

completed: sales
saved to: abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/sales
completed: customers
saved to: abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/customers
completed: products
saved to: abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/products
completed: stores
saved to: abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/stores


In [0]:
display(dbutils.fs.ls(f"{delta_path}/products"))

path,name,size,modificationTime
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/products/_delta_log/,_delta_log/,0,1792932961000
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/products/part-00000-2e5d7215-2b70-4a14-a5d5-28cef516cb5d.c000.snappy.parquet,part-00000-2e5d7215-2b70-4a14-a5d5-28cef516cb5d.c000.snappy.parquet,3632,1792934610000
abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/products/part-00000-7a6fda84-5ec9-47e5-beb5-d58befac0355.c000.snappy.parquet,part-00000-7a6fda84-5ec9-47e5-beb5-d58befac0355.c000.snappy.parquet,3632,1792932962000


### Creating Delta Live Tables

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS retailpulse;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retailpulse.bronze;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailpulse.bronze.sales
USING DELTA
LOCATION
'abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/sales';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retailpulse.bronze.customers
USING DELTA
LOCATION
'abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/customers';



CREATE TABLE IF NOT EXISTS retailpulse.bronze.products
USING DELTA
LOCATION
'abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/products';



CREATE TABLE IF NOT EXISTS retailpulse.bronze.stores
USING DELTA
LOCATION
'abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/delta/stores';

In [0]:
%sql
select * from retailpulse.bronze.customers
limit 10;

customer_id,customer_name,city,region,registration_date,_source_file,_ingestion_timestamp,_ingestion_batch_id
C0001,Customer 001,Hyderabad,South,2025-03-10,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0002,Customer 002,Bengaluru,South,2025-01-23,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0003,Customer 003,Chennai,South,2024-08-18,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0004,Customer 004,Mumbai,West,2024-07-19,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0005,Customer 005,Pune,West,2025-08-08,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0006,Customer 006,Delhi,North,2025-03-30,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0007,Customer 007,Jaipur,North,2024-03-14,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0008,Customer 008,Kolkata,East,2025-02-22,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0009,Customer 009,Bhubaneswar,East,2025-01-07,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125
C0010,Customer 010,Ahmedabad,West,2024-05-27,abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/customers/customers.csv,2026-10-25T13:23:23.302Z,1c2b7516-9230-45a0-8a78-e3c4b5d85125


In [0]:
%sql

select count(*) AS total_customers from retailpulse.bronze.customers;


total_customers
200


### Validation Checks

In [0]:
dataset_names = {
    "sales": sales_df,
    "customers": customers_df,
    "products": products_df,
    "stores": stores_df
}


for name, source_df in dataset_names.items():
    bronze_df = spark.read.format('delta').load(
        f"{delta_path}/{name}"
    )

    source_count = source_df.count()
    delta_count = bronze_df.count()


    print(
        "Dataset Name: ", name,
        "Source: ", source_count,
        "delta: ", delta_count,
        "Match", source_count == delta_count,
    )

Dataset Name:  sales Source:  1000 delta:  1000 Match True
Dataset Name:  customers Source:  200 delta:  200 Match True
Dataset Name:  products Source:  50 delta:  50 Match True
Dataset Name:  stores Source:  10 delta:  10 Match True


In [0]:
spark.read.format("delta").load(
    f"{delta_path}/sales"
).select(
    "_source_file",
    "_ingestion_timestamp",
    "_ingestion_batch_id"
).show(5, truncate=False)

+----------------------------------------------------------------------------------+--------------------------+------------------------------------+
|_source_file                                                                      |_ingestion_timestamp      |_ingestion_batch_id                 |
+----------------------------------------------------------------------------------+--------------------------+------------------------------------+
|abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales|2026-10-25 13:23:23.302127|1c2b7516-9230-45a0-8a78-e3c4b5d85125|
|abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales|2026-10-25 13:23:23.302127|1c2b7516-9230-45a0-8a78-e3c4b5d85125|
|abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales|2026-10-25 13:23:23.302127|1c2b7516-9230-45a0-8a78-e3c4b5d85125|
|abfss://retail-data@retailpulsestorageacc.dfs.core.windows.net/bronze/github/sales|2026-10-25 13:23:23.30

In [0]:
bronze_sales = spark.read.format("delta").load(
    f"{delta_path}/sales"
)

bronze_sales.filter(
    F.col("quantity").cast("int") < 0
).show()


+--------+----------+-----------+----------+--------+--------+----------+--------+---------+-------------------+--------------------+--------------------+--------------------+
|order_id|order_date|customer_id|product_id|store_id|quantity|unit_price|discount|   status| last_modified_date|        _source_file|_ingestion_timestamp| _ingestion_batch_id|
+--------+----------+-----------+----------+--------+--------+----------+--------+---------+-------------------+--------------------+--------------------+--------------------+
|  O00992|2025-12-21|      C0149|      P001|     S09|      -2|    199.00|  100.00|Completed|2025-12-21T12:25:00|abfss://retail-da...|2026-10-25 13:23:...|1c2b7516-9230-45a...|
+--------+----------+-----------+----------+--------+--------+----------+--------+---------+-------------------+--------------------+--------------------+--------------------+

